In [ ]:
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Connect and Query
try:
    conn = psycopg2.connect(
        dbname = "digital_banking_analytics",
        user = "postgres",
        password = "YOUR_PASSWORD",
        host = "127.0.0.1",
        port = "5432"
    )
    sql_query = """ WITH user_activity AS (
	SELECT
		users.user_id,
		DATE_TRUNC('month', users.signup_date) AS cohort_month,
		DATE_TRUNC('month', sessions.login_time) AS active_month,
		(EXTRACT(YEAR FROM sessions.login_time) - EXTRACT(YEAR FROM users.signup_date))
		* 12 + 
		(EXTRACT(MONTH FROM sessions.login_time) - EXTRACT(MONTH FROM users.signup_date)) AS month_number
	FROM users
	JOIN sessions
	ON users.user_id = sessions.user_id
    ),
    cohort_sizes AS (
        SELECT
            cohort_month,
            COUNT(DISTINCT user_id) AS total_cohort_size
        FROM user_activity
        WHERE month_number = 0
        GROUP BY cohort_month
    ),
    active_users_by_period AS (
        SELECT
            cohort_month,
            month_number,
            COUNT(DISTINCT user_id) AS active_users
        FROM user_activity
        GROUP BY cohort_month, month_number
    )

    SELECT
        a.cohort_month,
        c.total_cohort_size,
        a.month_number,
        active_users,
        ROUND(a.active_users::NUMERIC / c.total_cohort_size * 100, 2) AS retention_rate
    FROM active_users_by_period AS a
    JOIN cohort_sizes AS c
    ON a.cohort_month = c.cohort_month
    ORDER BY a.cohort_month, a.month_number; """

    df_cohort = pd.read_sql_query(sql_query, conn)
    conn.close()
    print("Data Loaded Successfully")

except Exception as e:
    print(f"Error: {e}")

df_cohort.head()


In [ ]:
# pivot tables

df_cohort['cohort_month'] = pd.to_datetime(df_cohort['cohort_month']).dt.strftime('%Y-%m')

new_pivot = df_cohort.pivot(
    index = 'cohort_month',
    columns = 'month_number',
    values = 'retention_rate'
)

new_pivot

In [ ]:
# plot heatmap
plt.figure(figsize = (10, 6))

sns.heatmap(
    data = new_pivot,
    annot = True,
    fmt = '.1f',
    cmap = 'Blues',
    linewidths = 0.5
)

plt.title("Digital Banking App Cohort Rettention Heatmap")
plt.xlabel("Months")
plt.ylabel("Group")

plt.savefig("../images/project-images/cohort_heatmap.png", dpi = 300, bbox_inches = "tight")

plt.show()

In [ ]:
import psycopg2
# transactions outlier detection
conn = psycopg2.connect(
    dbname = 'digital_banking_analytics',
    user = 'postgres',
    password = "YOUR_PASSWORD",
    host = '127.0.0.1',
    port = '5432'
)

txn_query = "SELECT type, amount FROM transactions"
df_tx = pd.read_sql_query(txn_query, conn)
conn.close()

print(df_tx['amount'].describe())
print()

Q3 = df_tx['amount'].quantile(0.75)
Q1 = df_tx['amount'].quantile(0.25)
IQR = Q3 - Q1
upper_bound = Q3 + (1.5 * IQR)

outliers = df_tx[df_tx['amount'] > upper_bound]
print(f"Number of Outliers: {outliers}")
print()
print(f"Percetage of Outliers: {(len(outliers)/len(df_tx) * 100):.2f}%")

In [ ]:
conn = psycopg2.connect(
    dbname = 'digital_banking_analytics',
    user = 'postgres',
    password = 'YOUR_PASSWORD',
    host = '127.0.0.1',
    port = '5432'
)

sql_query = """ 
SELECT
	u.user_id,
	EXTRACT(EPOCH FROM (MAX(CASE WHEN e.event_name = 'Account Created' THEN e.timestamp END) - 
	MIN(CASE WHEN e.event_name = 'Sign Up Started' THEN e.timestamp END))) AS onboarding_time_seconds,
	COUNT(DISTINCT t.transaction_id) AS transaction_count
FROM users u
JOIN events e
ON u.user_id = e.user_id
LEFT JOIN accounts a
ON u.user_id = a.user_id
LEFT JOIN transactions t
ON a.user_id = t.user_id
GROUP BY u.user_id;
"""
df_behaviour = pd.read_sql_query(sql_query, conn)
conn.close()

correlation = df_behaviour['onboarding_time_seconds'].corr(df_behaviour['transaction_count'])
print(correlation)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_784\1322306909.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_behaviour = pd.read_sql_query(sql_query, conn)


-0.02368883843963573
